# N-CMAPSS Data Exploration

This notebook explores the N-CMAPSS dataset structure, sensor distributions,
and RUL target statistics before training.

**References:**
- Arias Chao et al., *Aircraft Engine Run-to-Failure Dataset under Real Flight Conditions*, Data 2021.
- NASA dataset: https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/

In [1]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from cyclelayer.data.ncmapss import NCMAPSSDataset, SENSOR_NAMES_XS, OPERATING_CONDITION_NAMES
from cyclelayer.data.preprocessing import normalize, clip_rul

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

ModuleNotFoundError: No module named 'h5py'

## 1  HDF5 File Structure

In [ ]:
HDF5_PATH = Path('../data/NCMAPSS/N-CMAPSS_DS01-005.h5')

def print_hdf5_structure(path):
    with h5py.File(path, 'r') as f:
        def visitor(name, obj):
            indent = '  ' * name.count('/')
            shape = obj.shape if hasattr(obj, 'shape') else ''
            print(f"{indent}/{name}  {shape}")
        f.visititems(visitor)

print_hdf5_structure(HDF5_PATH)

## 2  Load Raw Arrays

In [ ]:
with h5py.File(HDF5_PATH, 'r') as f:
    W   = f['dev/W'][:]    # operating conditions
    X_s = f['dev/X_s'][:]  # measured sensors
    T   = f['dev/T'][:].ravel()  # RUL
    A   = f['dev/A'][:]    # auxiliary: unit | cycle | Fc | hs

print(f"W shape   : {W.shape}")
print(f"X_s shape : {X_s.shape}")
print(f"T shape   : {T.shape}")
print(f"A shape   : {A.shape}")
print(f"Unique units: {np.unique(A[:, 0].astype(int))}")

## 3  RUL Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(T, bins=80, color='steelblue', edgecolor='white')
axes[0].set_xlabel('RUL [cycles]')
axes[0].set_ylabel('Count')
axes[0].set_title('RUL distribution (raw)')

T_clipped = np.clip(T, 0, 125)
axes[1].hist(T_clipped, bins=80, color='coral', edgecolor='white')
axes[1].set_xlabel('RUL [cycles]')
axes[1].set_title('RUL distribution (clipped at 125)')

plt.tight_layout()
plt.show()

print(f"RUL  min={T.min():.0f}  max={T.max():.0f}  mean={T.mean():.1f}  median={np.median(T):.1f}")

## 4  Sensor Correlations

In [ ]:
df_sensors = pd.DataFrame(X_s, columns=SENSOR_NAMES_XS)
df_sensors['RUL'] = T

corr = df_sensors.corr()['RUL'].drop('RUL').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
corr.plot(kind='barh', ax=ax, color=np.where(corr > 0, 'steelblue', 'coral'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with RUL')
ax.set_title('Sensor–RUL correlations')
plt.tight_layout()
plt.show()

## 5  Single-Unit Degradation Trajectory

In [ ]:
unit_id = 1
mask = A[:, 0].astype(int) == unit_id

unit_sensors = X_s[mask]
unit_rul     = T[mask]
unit_cycle   = A[mask, 1]

# Plot selected sensors over flight cycles
sensors_to_plot = SENSOR_NAMES_XS[:6]
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)

for ax, name in zip(axes.flat, sensors_to_plot):
    idx = SENSOR_NAMES_XS.index(name)
    ax.plot(unit_cycle, unit_sensors[:, idx], linewidth=0.5, alpha=0.8)
    ax.set_ylabel(name)

for ax in axes[-1]:
    ax.set_xlabel('Flight cycle')

fig.suptitle(f'Sensor degradation – Unit {unit_id}', y=1.01)
plt.tight_layout()
plt.show()

## 6  Dataset Class Demo

In [ ]:
ds = NCMAPSSDataset(HDF5_PATH, split='dev', window_size=30)
print(f"Dataset size : {len(ds):,} windows")
print(f"n_features   : {ds.n_features}")

x, y = ds[0]
print(f"Window shape : {x.shape}")
print(f"RUL target   : {y.item():.1f}")